# Clase 13 — Word2Vec, Embeddings y Clustering de Texto (Versión Avanzada)
**Autor:** Josef Rodriguez

## Objetivos
- Entender las limitaciones semánticas de TF-IDF
- Entrenar un modelo Word2Vec
- Medir similitud entre palabras
- Visualizar embeddings en 2D, 3D y t-SNE
- Construir embeddings de documentos
- Aplicar clustering y evaluar K con silhouette


## Instalación sugerida
```bash
pip install gensim plotly scikit-learn pandas matplotlib
```


## Recordatorio conceptual

TF-IDF convierte texto en frecuencias ponderadas, pero no sabe que:

- `car` y `automobile` son parecidos
- `doctor` y `physician` están relacionados

Word2Vec resuelve esto aprendiendo vectores densos a partir del contexto.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import plotly.express as px

from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [ ]:
url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/SMSSpamCollection"

df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
df.head()


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean"] = df["text"].apply(clean_text)
sentences = [x.split() for x in df["clean"]]
len(sentences)


## Entrenar Word2Vec
Parámetros clave:
- `vector_size`: dimensión del embedding
- `window`: contexto
- `min_count`: frecuencia mínima
- `sg=1`: Skip-Gram


In [ ]:
w2v = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)
len(w2v.wv.index_to_key)


In [ ]:
for word in ["free", "call", "txt", "win", "love"]:
    if word in w2v.wv:
        print(f"\nSimilares a '{word}':")
        print(w2v.wv.most_similar(word, topn=5))


## Similaridad entre palabras


In [ ]:
pairs = [("free", "win"), ("call", "txt"), ("love", "please")]
for a, b in pairs:
    if a in w2v.wv and b in w2v.wv:
        print(a, b, "->", round(w2v.wv.similarity(a, b), 4))


## Visualización 2D con PCA


In [ ]:
words = w2v.wv.index_to_key[:120]
vectors = np.array([w2v.wv[w] for w in words])

pca2 = PCA(n_components=2)
coords2 = pca2.fit_transform(vectors)

plt.figure(figsize=(12, 8))
for i, word in enumerate(words):
    plt.scatter(coords2[i, 0], coords2[i, 1], alpha=0.7)
    plt.text(coords2[i, 0], coords2[i, 1], word, fontsize=8)
plt.title("Word Embeddings en 2D (PCA)")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()


## Visualización 3D interactiva con Plotly
Esta celda genera una gráfica girable, ideal para clase.


In [ ]:
words3 = w2v.wv.index_to_key[:100]
vectors3 = np.array([w2v.wv[w] for w in words3])

pca3 = PCA(n_components=3)
coords3 = pca3.fit_transform(vectors3)

plot_df = pd.DataFrame({
    "word": words3,
    "x": coords3[:, 0],
    "y": coords3[:, 1],
    "z": coords3[:, 2]
})

fig = px.scatter_3d(
    plot_df,
    x="x",
    y="y",
    z="z",
    text="word",
    hover_name="word",
    title="Embeddings Word2Vec en 3D"
)
fig.update_traces(marker=dict(size=4))
fig.show()


## t-SNE profesional para embeddings de palabras
t-SNE suele separar mejor estructuras locales que PCA.


In [ ]:
sample_words = w2v.wv.index_to_key[:200]
sample_vecs = np.array([w2v.wv[w] for w in sample_words])

tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
coords_tsne = tsne.fit_transform(sample_vecs)

plt.figure(figsize=(12, 8))
for i, word in enumerate(sample_words):
    plt.scatter(coords_tsne[i, 0], coords_tsne[i, 1], alpha=0.7)
    plt.text(coords_tsne[i, 0], coords_tsne[i, 1], word, fontsize=8)
plt.title("Embeddings Word2Vec con t-SNE")
plt.show()


## Embeddings de documentos
Promediaremos los embeddings de las palabras de cada SMS.


In [ ]:
def doc_vector(doc):
    tokens = doc.split()
    valid = [w2v.wv[w] for w in tokens if w in w2v.wv]
    if not valid:
        return np.zeros(w2v.vector_size)
    return np.mean(valid, axis=0)

X_doc = np.vstack(df["clean"].apply(doc_vector))
y = df["label"].map({"ham": 0, "spam": 1}).values

X_doc.shape


## Visualización de documentos con PCA


In [ ]:
doc_pca = PCA(n_components=2)
doc_coords = doc_pca.fit_transform(X_doc)

plt.figure(figsize=(10, 7))
plt.scatter(doc_coords[:, 0], doc_coords[:, 1], c=y, alpha=0.5)
plt.title("Documentos en 2D usando embeddings promedio")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()


## Visualización de documentos con t-SNE


In [ ]:
sample_n = min(2000, len(X_doc))
doc_tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
doc_coords_tsne = doc_tsne.fit_transform(X_doc[:sample_n])

plt.figure(figsize=(10, 7))
plt.scatter(doc_coords_tsne[:, 0], doc_coords_tsne[:, 1], c=y[:sample_n], alpha=0.5)
plt.title("Documentos con t-SNE")
plt.show()


## Elegir K con silhouette


In [ ]:
scores = []
k_values = range(2, 7)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_doc)
    scores.append(silhouette_score(X_doc, labels))

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), scores, marker="o")
plt.title("Silhouette Score por K")
plt.xlabel("K")
plt.ylabel("Silhouette")
plt.show()


## KMeans final


In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_doc)

plt.figure(figsize=(10, 7))
plt.scatter(doc_coords[:, 0], doc_coords[:, 1], c=clusters, alpha=0.5)
plt.title("Clusters de documentos con KMeans")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.show()


In [ ]:
pd.crosstab(
    pd.Series(y, name="label_real"),
    pd.Series(clusters, name="cluster")
)


## Conclusión
Word2Vec convierte palabras en geometría.

- palabras similares quedan cerca
- documentos pueden representarse como promedios
- clustering puede descubrir estructura interna

Esto prepara el camino para embeddings contextuales y transformers.
